# External data and wallet leakage

Issue #11. Do outside events explain *when* money leaves YAPEAL? Three candidate drivers, each against its YAPEAL counterpart from the SoW files:

- **crypto market** (bitcoin price, Fear & Greed index) vs outflows to Binance and Coinbase
- **interest rates** (SNB policy rate, savings deposit rate) vs outflows to banks (PostFinance, UBS, Kantonalbank)
- **Revolut's growth** vs Revolut top-ups, needs the manual Google Trends / Statista files in `data/external/manual/`

Fetch code in `src/external/`, sources in `docs/external_data.md`. Writes `data/external/external_monthly.parquet` (gitignored) and the figures in `figures/external/`.

This is about leakage over time, not churn: the churn label has no date (A2), so churn cannot be lined up with a month. The churn link is per customer, see `docs/leakage_churn.md`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src import plots, style
from src.external import fetch
from src.external.monthly import build_external_monthly, compare, critical_r, leakage_monthly, sources

style.apply()
pd.set_option("display.width", 200)

if not all((fetch.EXTERNAL / f"{name}.csv").exists() for name in ["bitcoin", "fear_greed", "snb_policy_rate", "snb_savings_rate", "fx"]):
    fetch.main()

ext = build_external_monthly()
leak = leakage_monthly()
both = ext.join(leak)
print(ext.shape, leak.shape, ext.index.min().date(), "to", ext.index.max().date())
sources()[["file", "source", "fetched_at"]]

## Pairs, side by side

One panel per series, no dual axes. Left the external series, right its YAPEAL counterpart, same months. Dotted lines are the events from `src/events.py`.

In [ ]:
PCT = 100

def pair_figure(rows, affects, name):
    fig, axes = plt.subplots(len(rows), 2, figsize=(11, 2.9 * len(rows)), squeeze=False)
    for (left, right), ax_row in zip(rows, axes):
        for (col, title, unit, scale), ax in zip((left, right), ax_row):
            ax.plot(both.index, both[col] * scale, color=style.C[0] if ax is ax_row[0] else style.C[1], lw=2)
            plots.month_axis(ax)
            style.clean(ax, title, unit)
            plots.add_event_markers(ax, affects, label=ax is ax_row[0])
    fig.tight_layout()
    plots.save_fig(fig, name, "external")
    return fig

pair_figure([
    (("btc_chf_mean", "bitcoin, monthly mean", "CHF", 1), ("crypto_exchanges_share", "to Binance + Coinbase, % of card spend", "%", PCT)),
    (("fear_greed", "crypto Fear & Greed index", "0 to 100", 1), ("crypto_exchanges_customers_pct", "customers paying crypto exchanges, % of active", "%", PCT)),
], "crypto", "01_crypto");

In [ ]:
pair_figure([
    (("snb_policy_rate", "SNB policy rate", "%", 1), ("traditional_banks_share", "to PostFinance, UBS, Kantonalbank, % of card spend", "%", PCT)),
    (("savings_deposit_rate", "savings deposit rate paid by banks", "%", 1), ("traditional_banks_customers_pct", "customers paying banks, % of active", "%", PCT)),
], "banks", "02_rates");

## Correlations

Levels of trending series correlate whatever the mechanism, so the change columns are the test: % change of prices, difference of rates and shares, external series leading by 0 to 2 months. With n = 35 monthly changes, |r| below the critical value is noise at the 5 % level.

In [ ]:
PAIRS = [
    ("btc_chf_mean", "crypto_exchanges_share"),
    ("btc_chf_mean", "crypto_exchanges_customers_pct"),
    ("fear_greed", "crypto_exchanges_share"),
    ("snb_policy_rate", "traditional_banks_share"),
    ("savings_deposit_rate", "traditional_banks_share"),
    ("savings_deposit_rate", "leak_all_share"),
    ("btc_chf_mean", "revolut_share"),
    ("eur_chf", "revolut_share"),
]
trend_cols = [c for c in ext.columns if c.startswith("trends_") and c != "trends_neon"]   # neon: broken topic, see docs
statista_cols = [c for c in ["revolut_mau_europe", "revolut_downloads_world"] if c in ext.columns]
revolut_drivers = trend_cols + statista_cols
PAIRS += [(c, y) for c in revolut_drivers for y in ("revolut_share", "revolut_customers_pct")]

print(f"critical |r|: monthly changes {critical_r(35):.2f}, quarterly changes {critical_r(11):.2f}")
compare(ext, leak, PAIRS)

In [ ]:
compare(ext, leak, PAIRS, lags=(0, 1), freq="Q")

## Rates: before and after the hikes

The policy rate changed five times, too few steps for a monthly change correlation. Mean per period instead.

In [ ]:
periods = {"negative rates, 2021-01 to 2022-05": ("2021-01", "2022-05"),
           "hikes, 2022-06 to 2022-12": ("2022-06", "2022-12"),
           "positive rates, 2023": ("2023-01", "2023-12")}
cols = {"snb_policy_rate": 1, "savings_deposit_rate": 1, "traditional_banks_share": PCT, "traditional_banks_customers_pct": PCT,
        "revolut_share": PCT, "crypto_exchanges_share": PCT, "leak_all_share": PCT}
pd.DataFrame({k: both.loc[a:z, list(cols)].mean() * pd.Series(cols) for k, (a, z) in periods.items()}).T.round(2)

## Revolut

Google Trends from `data/external/manual/google_trends.csv` (Revolut and Yuh, Switzerland, 100 = peak of the query; Neon left out, its topic series breaks) and two Statista series from AppMagic: Revolut monthly active users in Europe and app downloads worldwide. The yearly means show how fast Revolut grew against how little YAPEAL's Revolut leakage moved.

In [ ]:
rows = [("revolut_share", "to Revolut, % of card spend", "%", PCT), ("revolut_customers_pct", "customers topping up Revolut, % of active", "%", PCT)]
rows += [(c, f"Google Trends Switzerland: {c.removeprefix('trends_')}", "index", 1) for c in trend_cols]
labels = {"revolut_mau_europe": "Revolut monthly active users, Europe", "revolut_downloads_world": "Revolut app downloads per month, worldwide"}
rows += [(c, labels[c], "million", 1e-6) for c in statista_cols]
fig, axes = plots.small_multiples([(both[c] * s, t, u) for c, t, u, s in rows])
plots.save_fig(fig, "03_revolut", "external")
yearly = both.groupby(both.index.year)[revolut_drivers + ["revolut_share", "revolut_customers_pct"]].mean()
print("growth 2021 -> 2023:", (yearly.iloc[-1] / yearly.iloc[0] - 1).map("{:+.0%}".format).to_dict())
print("Trends Switzerland vs MAU Europe, r of monthly % changes:", round(both["trends_revolut"].pct_change().corr(both.get("revolut_mau_europe", both["trends_revolut"]).pct_change()), 2))
yearly.round(3)

## Findings

*(2026-10-02, API sources, Google Trends, Statista)*

- **crypto follows the market, with about a month's delay.** Monthly changes in the bitcoin price correlate with the change in outflows to Binance and Coinbase (r = 0.32 same month, 0.49 one month later), and with the share of customers paying them (0.43). Peaks in 2021Q2 and 2021Q4 match the bitcoin highs. 2022Q2 is the exception: prices fell and sentiment hit its low, outflows rose, buying the dip around Terra/Luna. In 2023 bitcoin recovered by half and crypto outflows kept falling, from 2.0 % of spend in 2021 to 0.9 % in 2023
- **bank outflows do not follow the rates.** The period means look like a rate effect (1.4 % of card spend under negative rates, 2.1 % during the hikes, 2.5 % in 2023), but the figure shows the rise started in the first half of 2021, from 0.7 % to 1.7 %, a year and a half before the first hike. The share of customers paying banks peaked in late 2021 (11 %), under negative rates. Rates may add to it in 2023, when banks started paying on savings (0.02 % to 0.76 %) and YAPEAL did not, but 36 months cannot separate that from the trend. Caveat A1: the bank rows may partly be ATM withdrawals
- **Revolut grew four times as fast as its share of YAPEAL wallets.** Revolut's monthly active users in Europe went from 2.0 m (2021 mean) to 5.0 m (2023, +150 %), downloads worldwide from 0.8 m to 2.0 m a month, Swiss search interest +58 %. YAPEAL's card spend going to Revolut rose by a third (2.9 % → 3.8 %) and the share of active customers topping up Revolut did not move (about 5 %). Levels correlate (r ≈ 0.7, all trending up), monthly changes do not (r = 0.07 for spend share). Swiss search interest and European MAU move together month by month (r = 0.55), so Trends is a fair Swiss proxy
- **reading:** YAPEAL is not losing its card base to Revolut at Revolut's pace. Leakage to Revolut sits with a stable group, about one in twenty active customers, who send more each year. The customers to talk to are known and few, which fits the "online & top-ups" segment
- **Revolut as the holiday card.** Search interest peaks in July 2022 and July 2023 (100 and 97), and so does Revolut's share of YAPEAL spend (4.4 % and 4.8 % against yearly means of 3.7 % and 3.8 %). No summer peak in 2021, the year without much travel. Yuh and Neon show no link
- n = 36 months: association, not cause, and one shared trend can produce several of these correlations